In [1]:
from pathlib import Path
import os, re, random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.home() / "Desktop/Concordia/Winter 2026/COMP 472/Project"

# Raw dataset root
RAW_ROOT = PROJECT_ROOT / "raw_data/car-brand-classification-dataset"

# Output root
OUT_ROOT = PROJECT_ROOT / "processed_data/car_brand_classification"

# Existing map used before
CLASS_MAP_PATH = PROJECT_ROOT / "processed_data/stanford_cars/class_map.csv"

# New "general" map output (appended)
GENERAL_MAP_PATH = PROJECT_ROOT / "processed_data/class_map_general.csv"

IMG_SIZE = 224
SEED = 42

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def pil_open_rgb(path: Path) -> Image.Image:
    # open + apply EXIF rotation + force RGB (fixes 1-channel & RGBA)
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        # If palette/transparency, convert safely:
        if im.mode in ("P", "LA", "RGBA"):
            im = im.convert("RGBA").convert("RGB")
        else:
            im = im.convert("RGB")
        return im

def resize_with_padding(im: Image.Image, size: int) -> Image.Image:
    w, h = im.size
    if w == 0 or h == 0:
        raise ValueError("Invalid image with zero dimension")

    scale = size / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    im = im.resize((new_w, new_h), Image.BICUBIC)

    pad_w = size - new_w
    pad_h = size - new_h
    left = pad_w // 2
    top = pad_h // 2
    right = pad_w - left
    bottom = pad_h - top
    im = ImageOps.expand(im, border=(left, top, right, bottom), fill=(0, 0, 0))
    return im

def safe_filename(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def normalize_brand(s: str) -> str:
    return str(s).strip().lower()

set_seed(SEED)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_OUT = OUT_ROOT / "images_processed"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT)
print("CLASS_MAP_PATH:", CLASS_MAP_PATH, "| exists:", CLASS_MAP_PATH.exists())
print("GENERAL_MAP_PATH:", GENERAL_MAP_PATH)

PROJECT_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project
RAW_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/raw_data/car-brand-classification-dataset | exists: True
OUT_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification
CLASS_MAP_PATH: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/class_map.csv | exists: True
GENERAL_MAP_PATH: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/class_map_general.csv


Load Stanford class_map

In [2]:
assert CLASS_MAP_PATH.exists(), f"Missing class map: {CLASS_MAP_PATH}"

class_map_df = pd.read_csv(CLASS_MAP_PATH)

# Expect at least: brand, brand_id
expected_cols = {"brand", "brand_id"}
missing_cols = expected_cols - set(class_map_df.columns)
assert not missing_cols, f"class_map.csv missing columns: {missing_cols}"

class_map_df["brand_norm"] = class_map_df["brand"].apply(normalize_brand)
brand_to_id = dict(zip(class_map_df["brand_norm"], class_map_df["brand_id"]))

max_existing_id = int(class_map_df["brand_id"].max())
print("Loaded brands in existing map:", len(brand_to_id))
print("Max existing brand_id:", max_existing_id)

Loaded brands in existing map: 49
Max existing brand_id: 48


Scan raw dataset to combine raw train/val/test

In [3]:
assert RAW_ROOT.exists(), f"RAW_ROOT not found: {RAW_ROOT}"

# Dataset has these split folders: train / val / test
split_dirs = [RAW_ROOT / "train", RAW_ROOT / "val", RAW_ROOT / "test"]
for d in split_dirs:
    assert d.exists(), f"Expected split folder missing: {d}"

rows = []
for split_dir in split_dirs:
    # inside each split folder: <BrandName>/imagefiles...
    for brand_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        brand = brand_dir.name  # e.g., "Mercedes-Benz"
        imgs = [p for p in brand_dir.rglob("*") if p.is_file() and is_image_file(p)]
        for p in imgs:
            rows.append({
                "img_path": str(p),
                "brand_raw": brand,
            })

df_raw = pd.DataFrame(rows)
print("Total raw images found (train+val+test combined):", len(df_raw))
print("Unique brands found:", df_raw["brand_raw"].nunique())
print("Brands:", sorted(df_raw["brand_raw"].unique().tolist()))
df_raw.head()

Total raw images found (train+val+test combined): 16467
Unique brands found: 33
Brands: ['Acura', 'Aston Martin', 'Audi', 'BMW', 'Bentley', 'Buick', 'Cadillac', 'Chevrolet', 'Chrysler', 'Dodge', 'FIAT', 'Ford', 'GMC', 'Honda', 'Hyundai', 'INFINITI', 'Jaguar', 'Jeep', 'Kia', 'Land Rover', 'Lexus', 'Lincoln', 'MINI', 'Mazda', 'Mercedes-Benz', 'Mitsubishi', 'Nissan', 'Porsche', 'Ram', 'Subaru', 'Toyota', 'Volkswagen', 'Volvo']


,img_path,brand_raw
0,/Users/pegahaghili/Desktop/Concordia/Winter 20...,Acura
1,/Users/pegahaghili/Desktop/Concordia/Winter 20...,Acura
2,/Users/pegahaghili/Desktop/Concordia/Winter 20...,Acura
3,/Users/pegahaghili/Desktop/Concordia/Winter 20...,Acura
4,/Users/pegahaghili/Desktop/Concordia/Winter 20...,Acura


Normalize brand names + check missing in map

In [4]:
def normalize_brand(s: str) -> str:
    return str(s).strip().lower()


BRAND_ALIAS = {
    "mercedes benz": "mercedes-benz"
}

def map_brand(brand_raw: str) -> str:
    b = normalize_brand(brand_raw)
    b = BRAND_ALIAS.get(b, b)
    return b

# Normalize dataset brands
df_raw["brand"] = df_raw["brand_raw"].apply(map_brand)

# Normalize existing class_map brands
class_map_df["brand_norm"] = class_map_df["brand"].apply(normalize_brand)
brand_to_id = dict(zip(class_map_df["brand_norm"], class_map_df["brand_id"]))

dataset_brands = sorted(df_raw["brand"].unique().tolist())
missing_in_map = sorted(set(dataset_brands) - set(brand_to_id.keys()))

print("Dataset brands (normalized):", dataset_brands)
print("-------- Missing from existing map (case-insensitive):", missing_in_map)

Dataset brands (normalized): ['acura', 'aston martin', 'audi', 'bentley', 'bmw', 'buick', 'cadillac', 'chevrolet', 'chrysler', 'dodge', 'fiat', 'ford', 'gmc', 'honda', 'hyundai', 'infiniti', 'jaguar', 'jeep', 'kia', 'land rover', 'lexus', 'lincoln', 'mazda', 'mercedes-benz', 'mini', 'mitsubishi', 'nissan', 'porsche', 'ram', 'subaru', 'toyota', 'volkswagen', 'volvo']
-------- Missing from existing map (case-insensitive): ['kia', 'lexus', 'subaru']


Creating general map

In [5]:
general_map_df = class_map_df[["brand", "brand_id"]].copy()
general_map_df["brand_norm"] = general_map_df["brand"].apply(normalize_brand)

if missing_in_map:
    next_id = int(general_map_df["brand_id"].max()) + 1
    new_rows = []
    for b in missing_in_map:
        new_rows.append({"brand": b, "brand_id": next_id, "brand_norm": b})
        next_id += 1

    general_map_df = pd.concat([general_map_df, pd.DataFrame(new_rows)], ignore_index=True)

# Keep unique by brand_norm
general_map_df = (
    general_map_df.sort_values("brand_id")
    .drop_duplicates(subset=["brand_norm"], keep="first")
    .reset_index(drop=True)
)

general_map_df[["brand", "brand_id"]].to_csv(GENERAL_MAP_PATH, index=False)
print("Saved general map:", GENERAL_MAP_PATH)
print("General map brand count:", len(general_map_df))

# Use the general map for THIS dataset going forward
general_brand_to_id = dict(zip(general_map_df["brand_norm"], general_map_df["brand_id"]))

✅ Saved general map: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/class_map_general.csv
General map brand count: 52


Attach brand_id using the GENERAL map

In [6]:
df = df_raw[["img_path", "brand"]].copy()
df["brand_id"] = df["brand"].map(general_brand_to_id)

still_missing = df[df["brand_id"].isna()]["brand"].unique().tolist()
assert len(still_missing) == 0, f"Some brands still missing after general map: {still_missing}"

df["brand_id"] = df["brand_id"].astype(int)

print("Rows ready for processing:", len(df))
df.head()

Rows ready for processing: 16467


,img_path,brand,brand_id
0,/Users/pegahaghili/Desktop/Concordia/Winter 20...,acura,0
1,/Users/pegahaghili/Desktop/Concordia/Winter 20...,acura,0
2,/Users/pegahaghili/Desktop/Concordia/Winter 20...,acura,0
3,/Users/pegahaghili/Desktop/Concordia/Winter 20...,acura,0
4,/Users/pegahaghili/Desktop/Concordia/Winter 20...,acura,0


Clean + resize images

In [7]:
bad_files = []
records = []

for r in tqdm(df.itertuples(index=False), total=len(df), desc="Cleaning Car Brand Classification"):
    src = Path(r.img_path)
    if not src.exists():
        bad_files.append({"file": str(src), "error": "File not found"})
        continue

    try:
        im = pil_open_rgb(src)
        im = resize_with_padding(im, IMG_SIZE)

        brand_folder = safe_filename(r.brand)
        out_dir = IMAGES_OUT / brand_folder
        out_dir.mkdir(parents=True, exist_ok=True)

        out_path = out_dir / (src.stem + ".jpg")
        im.save(out_path, format="JPEG", quality=95, optimize=True)

        rel_path = str(out_path.relative_to(PROJECT_ROOT)).replace("\\", "/")

        records.append({
            "path": rel_path,
            "brand": r.brand,
            "brand_id": int(r.brand_id),
        })

    except Exception as e:
        bad_files.append({"file": str(src), "error": str(e)})

manifest = pd.DataFrame(records)
bad_df = pd.DataFrame(bad_files)

print("Processed:", len(manifest))
print("Bad files:", len(bad_df))

manifest_path = OUT_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

if len(bad_df) > 0:
    bad_df.to_csv(OUT_ROOT / "bad_files.csv", index=False)

print("Saved manifest:", manifest_path)
manifest.head()

Cleaning Car Brand Classification: 100%|██████████| 16467/16467 [01:52<00:00, 146.31it/s]


Processed: 16467
Bad files: 0
Saved manifest: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/manifest.csv


,path,brand,brand_id
0,processed_data/car_brand_classification/images...,acura,0
1,processed_data/car_brand_classification/images...,acura,0
2,processed_data/car_brand_classification/images...,acura,0
3,processed_data/car_brand_classification/images...,acura,0
4,processed_data/car_brand_classification/images...,acura,0


Save dataset-specific class_map.csv (subset)

In [8]:
class_map_out = (
    manifest[["brand", "brand_id"]]
    .drop_duplicates()
    .sort_values("brand_id")
    .reset_index(drop=True)
)

class_map_out_path = OUT_ROOT / "class_map.csv"
class_map_out.to_csv(class_map_out_path, index=False)
print("Saved dataset class map:", class_map_out_path)
class_map_out.head()

Saved dataset class map: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/class_map.csv


,brand,brand_id
0,acura,0
1,aston martin,2
2,audi,3
3,bentley,4
4,bmw,5


Stats

In [9]:
brand_counts = (
    manifest.groupby(["brand_id", "brand"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

brand_counts_path = OUT_ROOT / "brand_counts.csv"
brand_counts.to_csv(brand_counts_path, index=False)
print("Saved:", brand_counts_path)

stats_lines = [
    "Car Brand Classification Dataset (ahmedelsany) - Preprocessing Stats",
    f"Total processed images: {len(manifest)}",
    f"Unique brands: {manifest['brand'].nunique()}",
    f"Target image size: {IMG_SIZE}x{IMG_SIZE}",
    f"Bad/corrupt/missing images removed: {len(bad_df)}",
    f"Existing base map used: {CLASS_MAP_PATH}",
    f"General map saved at: {GENERAL_MAP_PATH}",
    f"Dataset class map saved at: {class_map_out_path}",
]

(OUT_ROOT / "stats_full.txt").write_text("\n".join(stats_lines), encoding="utf-8")
print("\n".join(stats_lines))

Saved: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/brand_counts.csv
Car Brand Classification Dataset (ahmedelsany) - Preprocessing Stats
Total processed images: 16467
Unique brands: 33
Target image size: 224x224
Bad/corrupt/missing images removed: 0
Existing base map used: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/class_map.csv
General map saved at: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/class_map_general.csv
Dataset class map saved at: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/class_map.csv


Sample images (3 per brand) like before

In [10]:
SAMPLES_ROOT = PROJECT_ROOT / "processed_data/samples/car_brand_classification"
SAMPLES_ROOT.mkdir(parents=True, exist_ok=True)

samples_per_brand = 3

for brand, group in manifest.groupby("brand"):
    out_dir = SAMPLES_ROOT / safe_filename(brand)
    out_dir.mkdir(parents=True, exist_ok=True)

    sample_rows = group.sample(n=min(samples_per_brand, len(group)), random_state=SEED)

    for i, r in enumerate(sample_rows.itertuples(index=False), start=1):
        src = PROJECT_ROOT / r.path
        dst = out_dir / f"sample_{i}.jpg"
        dst.write_bytes(src.read_bytes())

print("Sample images created at:", SAMPLES_ROOT)

Sample images created at: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/samples/car_brand_classification


Split into train/val/test (80/10/10)

In [11]:
manifest_full = pd.read_csv(OUT_ROOT / "manifest.csv")

# Train (80%) vs Temp (20%)
train_df, temp_df = train_test_split(
    manifest_full,
    test_size=0.20,
    stratify=manifest_full["brand"],
    random_state=SEED
)

# Val (10%) vs Test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["brand"],
    random_state=SEED
)

train_path = OUT_ROOT / "manifest_train.csv"
val_path   = OUT_ROOT / "manifest_val.csv"
test_path  = OUT_ROOT / "manifest_test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved:")
print(" -", train_path, "| rows:", len(train_df))
print(" -", val_path,   "| rows:", len(val_df))
print(" -", test_path,  "| rows:", len(test_df))

# Distribution check
print("\nTrain distribution (top 10):\n", train_df["brand"].value_counts().head(10))
print("\nVal distribution (top 10):\n", val_df["brand"].value_counts().head(10))
print("\nTest distribution (top 10):\n", test_df["brand"].value_counts().head(10))

Saved:
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/manifest_train.csv | rows: 13173
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/manifest_val.csv | rows: 1647
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/car_brand_classification/manifest_test.csv | rows: 1647

Train distribution (top 10):
 brand
dodge           400
volkswagen      400
nissan          400
jeep            400
chrysler        400
infiniti        400
aston martin    399
mitsubishi      399
ford            399
volvo           399
Name: count, dtype: int64

Val distribution (top 10):
 brand
hyundai       50
volkswagen    50
fiat          50
subaru        50
lexus         50
volvo         50
bmw           50
gmc           50
chevrolet     50
ram           50
Name: count, dtype: int64

Test distribution (top 10):
 brand
buick         50
kia           50
bmw     